# 2pt

In [11]:
import h5py,os,re,click
import numpy as np
from itertools import permutations
from sympy.combinatorics import Permutation

ens='cD211.054.96'

path='data_aux/cfgs_run'
with open(path,'r') as f:
    cfgs=f.read().splitlines()
    
id=np.eye(4)
gamma_1=gamma_x=np.array([[0.,0.,0.,1j],[0.,0.,1j,0.],[0.,-1j,0.,0.],[-1j,0.,0.,0.]])
gamma_2=gamma_y=np.array([[0.,0.,0.,1.],[0.,0.,-1.,0.],[0.,-1.,0.,0.],[1.,0.,0.,0.]])
gamma_3=gamma_z=np.array([[0.,0.,1j,0.],[0.,0.,0.,-1j],[-1j,0.,0.,0.],[0.,1j,0.,0.]])
gamma_4=gamma_t=np.array([[1.,0.,0.,0.],[0.,1.,0.,0.],[0.,0.,-1.,0.],[0.,0.,0.,-1.]])
gamma_5=(gamma_1@gamma_2@gamma_3@gamma_4)
#
P0=(id+gamma_4)/4; Px=1j*gamma_5@gamma_1@P0; Py=1j*gamma_5@gamma_2@P0; Pz=1j*gamma_5@gamma_3@P0
P0n=(id-gamma_4)/4; Pxn=1j*gamma_5@gamma_1@P0n; Pyn=1j*gamma_5@gamma_2@P0n; Pzn=1j*gamma_5@gamma_3@P0n
#
# coeff_{i} C_{i=4a+b} = Tr[proj@C] = proj_{ba} C_{ab} = proj_{ba} C_{4a+b} 
# coeff_{i=4a+b} = proj_{ba}
dirac2proj=np.array([[complex(ele) for row in proj.T for ele in row] for proj in [P0,Px,Py,Pz]])[:,[0,1,4,5]]
dirac2proj_bw=np.array([[complex(ele) for row in proj.T for ele in row] for proj in [P0n,Pxn,Pyn,Pzn]])[:,[10,11,14,15]]
    
moms=None

key2dat={}
for icfg,cfg in enumerate(cfgs):
    print(f'{icfg}/{len(cfgs)}',end='           \r')
    path=f'/p/project1/ngff/li47/code/projectData/02_discNJN_1D/cD211.054.96/data_post_hold/{cfg}/'
    files=[file for file in os.listdir(path) if file.startswith('N')]
    
    dic={}
    for file in files:
        with h5py.File(f'{path}{file}') as f:
            if moms is None:
                moms=f['moms'][:]
            
            for td in ['data','data_bw']:
                for fla in ['N1_N1','N2_N2']:
                    key=(td,fla)
                    if key not in dic:
                        dic[key]=[]
                        
                    for src in f[td].keys():
                        dic[key].append(f[f'{td}/{src}/{fla}'][:])
                        
    for td in ['data','data_bw']:
        for fla in ['N1_N1','N2_N2']:
            key=(td,fla)
            if key not in key2dat:
                key2dat[key]=[]
            
            t=np.mean(dic[key],axis=0)
            
            key2dat[key].append(t)

path=f'/p/project1/ngff/li47/code/projectData/05_moments/{ens}/data_merge/disc_2pt_vevsub.h5'
with h5py.File(path,'w') as f:
    f.create_dataset('notes',data=['cfg,time,mom,proj'])
    f.create_dataset('moms',data=moms)
    for td in ['data','data_bw']:
        for fla in ['N1_N1','N2_N2']:
            key=(td,fla)
            t=key2dat[key]
            if td in ['data']:
                t=np.einsum('pd,ctmd->ctmp',dirac2proj,t)
            else:
                t=np.einsum('pd,ctmd->ctmp',dirac2proj_bw,t)
            f.create_dataset(f'{td}/{fla}',data=t)

# jvev